# Cargar datos - Proyecto Integrador M5 Avance 1

Este notebook corresponde al **avance 1: Versionamiento y Colaboracion**. Su funcion es cargar la base no productiva del proyecto, validar que el archivo tiene el esquema esperado y dejar evidencia inicial de calidad antes del EDA.

El caso del PI es financiero: se usa informacion historica de creditos para predecir si una solicitud tendra `Pago_atiempo`.

## 1. Configuracion e importacion de librerias

Se cargan pandas, graficos y las funciones del pipeline. Usar las funciones del proyecto evita que el notebook limpie los datos de una forma y el modelo de otra.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
if (PROJECT_DIR / "mlops_pipeline").exists():
    PROJECT_DIR = PROJECT_DIR / "mlops_pipeline"
SRC_DIR = PROJECT_DIR / "src"
sys.path.insert(0, str(SRC_DIR))

from ft_engineering import (
    DATE_COLUMN,
    REQUIRED_COLUMNS,
    TARGET,
    add_features,
    clean_data,
    load_data,
)

DATA_PATH = PROJECT_DIR / "Base_de_datos.csv"
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid")

## 2. Ubicacion del dataset

La consigna del avance indica usar un dataset de ejemplo en CSV. En este repositorio se usa `Base_de_datos.csv`, generado desde la base oficial recibida para el PI.

In [2]:
print(DATA_PATH)
assert DATA_PATH.exists(), f"No existe el archivo: {DATA_PATH}"

C:\Users\Nassi\OneDrive\Desktop\ProyectoM5_Nassim\mlops_pipeline\Base_de_datos.csv


## 3. Carga inicial

Primero se carga la base cruda para revisar dimensiones, columnas y una muestra de registros.

In [3]:
raw_df = load_data(DATA_PATH)
print(f"Filas: {raw_df.shape[0]:,}")
print(f"Columnas: {raw_df.shape[1]:,}")
raw_df.head()

Filas: 10,763
Columnas: 23


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,puntaje_datacredito,cant_creditosvigentes,huella_consulta,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,7,"45,647.4803","3,692,160.0000",10,42,Independiente,8000000,2500000,341296,88.7681,695.0000,10,5,0.0000,"51,258.0000","51,258.0000",0.0000,5,0,0,"908,526.0000",Estable,1
1,4,"45,769.4080","840,000.0000",6,60,Empleado,3000000,2000000,124876,95.2278,789.0000,3,1,0.0000,"8,673.0000","8,673.0000",0.0000,0,0,2,"939,017.0000",Creciente,1
2,9,"46,030.5157","5,974,028.4000",10,36,Independiente,4036000,829000,529554,47.6139,740.0000,4,5,0.0000,"18,702.0000","18,702.0000",0.0000,3,0,0,NaN,NaN,0
3,4,"45,873.5029","1,671,240.0000",6,48,Empleado,1524547,498000,252420,95.2278,837.0000,4,4,0.0000,"15,782.0000","15,782.0000",0.0000,3,0,0,"1,536,193.0000",Creciente,1
4,9,"45,773.4753","2,781,636.0000",11,44,Empleado,5000000,4000000,217037,95.2278,771.0000,4,6,0.0000,"204,804.0000","204,804.0000",0.0000,3,0,1,"933,473.0000",Creciente,1


## 4. Validacion del esquema minimo

Se verifica que esten todas las columnas necesarias. Si falta alguna, el pipeline debe detenerse porque el modelo no podria entrenar de forma coherente.

In [4]:
missing_columns = REQUIRED_COLUMNS.difference(raw_df.columns)
extra_columns = set(raw_df.columns).difference(REQUIRED_COLUMNS)
print("Columnas faltantes:", sorted(missing_columns))
print("Columnas extra:", sorted(extra_columns))
assert not missing_columns

Columnas faltantes: []
Columnas extra: []


## 5. Tipos de datos crudos

Esta revision muestra como llegan los datos antes de la limpieza. Algunas columnas numericas pueden venir como texto o con nulos; la fecha viene como serial de Excel y se convertira luego a fecha real.

In [5]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10763 entries, 0 to 10762
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   tipo_credito                   10763 non-null  int64  
 1   fecha_prestamo                 10763 non-null  float64
 2   capital_prestado               10763 non-null  float64
 3   plazo_meses                    10763 non-null  int64  
 4   edad_cliente                   10763 non-null  int64  
 5   tipo_laboral                   10763 non-null  str    
 6   salario_cliente                10763 non-null  int64  
 7   total_otros_prestamos          10763 non-null  int64  
 8   cuota_pactada                  10763 non-null  int64  
 9   puntaje                        10763 non-null  float64
 10  puntaje_datacredito            10757 non-null  float64
 11  cant_creditosvigentes          10763 non-null  int64  
 12  huella_consulta                10763 non-null  int64  
 1

## 6. Nulos y duplicados antes de limpiar

Se cuantifican valores faltantes y registros duplicados para saber si la base necesita limpieza. No se eliminan filas completas por tener nulos, porque varias columnas financieras pueden imputarse durante el modelado.

In [6]:
null_summary = (
    raw_df.isna().sum().to_frame("nulos")
    .assign(pct_nulos=lambda data: (data["nulos"] / len(raw_df) * 100).round(2))
    .sort_values("nulos", ascending=False)
)
print("Duplicados exactos:", raw_df.duplicated().sum())
null_summary[null_summary["nulos"] > 0]

Duplicados exactos: 0


,nulos,pct_nulos
tendencia_ingresos,2932,27.2400
promedio_ingresos_datacredito,2930,27.2200
saldo_mora_codeudor,590,5.4800
saldo_principal,405,3.7600
saldo_mora,156,1.4500
saldo_total,156,1.4500
puntaje_datacredito,6,0.0600


## 7. Limpieza inicial reutilizable

Se aplica la limpieza definida en `ft_engineering.py`: unifica nulos, corrige tipos, convierte la fecha, crea un identificador tecnico `loan_id` y normaliza categorias.

In [7]:
clean_df = clean_data(raw_df)
print(clean_df.shape)
clean_df.head()

(10763, 24)


,loan_id,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,puntaje_datacredito,cant_creditosvigentes,huella_consulta,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,REQ000000,7,2024-12-21 11:31:34.999999368,"3,692,160.0000",10,42,Independiente,8000000,2500000,341296,88.7681,695.0000,10,5,0.0000,"51,258.0000","51,258.0000",0.0000,5,0,0,"908,526.0000",Estable,1
1,REQ000001,4,2025-04-22 09:47:34.999999768,"840,000.0000",6,60,Empleado,3000000,2000000,124876,95.2278,789.0000,3,1,0.0000,"8,673.0000","8,673.0000",0.0000,0,0,2,"939,017.0000",Creciente,1
2,REQ000002,9,2026-01-08 12:22:39.999999823,"5,974,028.4000",10,36,Independiente,4036000,829000,529554,47.6139,740.0000,4,5,0.0000,"18,702.0000","18,702.0000",0.0000,3,0,0,NaN,Desconocido,0
3,REQ000003,4,2025-08-04 12:04:09.999999972,"1,671,240.0000",6,48,Empleado,1524547,498000,252420,95.2278,837.0000,4,4,0.0000,"15,782.0000","15,782.0000",0.0000,3,0,0,"1,536,193.0000",Creciente,1
4,REQ000004,9,2025-04-26 11:24:26.000000291,"2,781,636.0000",11,44,Empleado,5000000,4000000,217037,95.2278,771.0000,4,6,0.0000,"204,804.0000","204,804.0000",0.0000,3,0,1,"933,473.0000",Creciente,1


## 8. Validaciones despues de limpiar

Estas reglas comprueban que el dataset queda listo para EDA y modelado: objetivo binario, identificador unico, fechas validas y rangos basicos coherentes.

In [8]:
checks = {
    "objetivo_binario": set(clean_df[TARGET].dropna().unique()).issubset({0, 1}),
    "loan_id_unico": clean_df["loan_id"].is_unique,
    "fecha_valida": clean_df[DATE_COLUMN].notna().all(),
    "plazo_positivo": (clean_df["plazo_meses"] >= 1).all(),
    "edad_minima": (clean_df["edad_cliente"] >= 18).all(),
    "capital_no_negativo": (clean_df["capital_prestado"] >= 0).all(),
}
pd.Series(checks, name="validacion")

objetivo_binario       True
loan_id_unico          True
fecha_valida           True
plazo_positivo         True
edad_minima            True
capital_no_negativo    True
Name: validacion, dtype: bool

## 9. Distribucion de la variable objetivo

`Pago_atiempo = 1` significa que el credito pago a tiempo. La clase positiva domina la muestra, por eso en modelado no basta mirar accuracy: tambien se revisan ROC-AUC, balanced accuracy, recall y matriz de confusion.

In [9]:
target_distribution = clean_df[TARGET].value_counts().rename("conteo").to_frame()
target_distribution["porcentaje"] = (target_distribution["conteo"] / len(clean_df) * 100).round(2)
target_distribution

,conteo,porcentaje
Pago_atiempo,,
1,10252,95.2500
0,511,4.7500


## 10. Dataset con features derivadas

Aunque el avance 1 se enfoca en carga y EDA, se deja una primera vista de las variables derivadas que usaran avances posteriores.

In [10]:
featured_df = add_features(clean_df)
new_columns = [col for col in featured_df.columns if col not in clean_df.columns]
print(new_columns)
featured_df[new_columns].head()

['prestamo_year', 'prestamo_month', 'cuota_salario_ratio', 'capital_salario_ratio', 'otros_prestamos_salario_ratio', 'carga_total_salario_ratio', 'capital_por_mes', 'mora_ratio', 'saldo_principal_ratio', 'has_mora', 'has_codeudor_mora', 'creditos_total', 'datacredito_income_gap']


,prestamo_year,prestamo_month,cuota_salario_ratio,capital_salario_ratio,otros_prestamos_salario_ratio,carga_total_salario_ratio,capital_por_mes,mora_ratio,saldo_principal_ratio,has_mora,has_codeudor_mora,creditos_total,datacredito_income_gap
0,2024,12,0.0427,0.4615,0.3125,0.3552,"369,216.0000",0.0000,1.0000,0,0,5,"-7,091,474.0000"
1,2025,4,0.0416,0.2800,0.6667,0.7083,"140,000.0000",0.0000,1.0000,0,0,2,"-2,060,983.0000"
2,2026,1,0.1312,1.4802,0.2054,0.3366,"597,402.8400",0.0000,1.0000,0,0,3,NaN
3,2025,8,0.1656,1.0962,0.3267,0.4922,"278,540.0000",0.0000,1.0000,0,0,3,"11,646.0000"
4,2025,4,0.0434,0.5563,0.8000,0.8434,"252,876.0000",0.0000,1.0000,0,0,4,"-4,066,527.0000"


## 11. Conclusiones de carga

- La base correcta contiene 10,763 registros y 23 columnas originales.
- La variable objetivo del PI es `Pago_atiempo`.
- No se detectan duplicados exactos relevantes.
- Hay nulos en variables financieras y de tendencia; se unifican como `NaN` o `Desconocido` segun el tipo.
- La fecha del prestamo se transforma desde serial de Excel a fecha real.
- El archivo queda listo para el notebook de comprension EDA.